# MobileNetV3-Large phone-demo model (MAAI901 skin-lesion project)

Trains the MobileNetV3-Large classifier used in the phone demo, with the **same recipe as the group report** (full fine-tuning, 8 classes, ISIC 2019 + PAD-UFES-20). It then evaluates the model and exports a small phone-ready file.

**What "Run All" does**
1. Downloads ISIC 2019 from the official ISIC link (about 9 GB) and unpacks your PAD-UFES-20 download.
2. Prepares the images like the report: 8 classes, identical files handled, and a 70/15/15 split by patient, lesion and duplicate group, so no person or lesion appears in two splits.
3. Fine-tunes MobileNetV3-Large with the report's settings.
4. Evaluates on the held-out test split and compares the result with the report's MobileNetV3-Large.
5. Exports `phone_demo_export.zip`: the model (about 12 MB), its settings and a few sample test images for the phone page.

**Before running:** follow `README.txt`. You need the PAD-UFES-20 zip and three extra packages. Everything stays in this folder; nothing goes to Google Drive.

**Stopping is safe.** Rerun the notebook and downloads resume, finished steps are skipped and training continues from the last finished epoch.

## 1. Settings and checks

In [ ]:
import os, sys, io, json, math, time, random, hashlib, zipfile, shutil, platform, threading, urllib.request, urllib.error
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd
from PIL import Image, ImageOps
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision, torchvision.transforms as T
from torchvision.transforms import InterpolationMode
import matplotlib.pyplot as plt


class Stop(Exception):
    """Stops 'Run All' on purpose, with a message saying what to do."""


TEST_MODE = os.environ.get('DEMO_TEST') == '1'   # tiny offline check of this notebook only; leave it unset

# Same settings as the group's final training notebook (Final_MAAI901_Training_Evaluation.ipynb)
SEED = 42
CLASSES = ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC']
CLASS_NAMES = {'MEL': 'Melanoma', 'NV': 'Melanocytic naevus (mole)', 'BCC': 'Basal cell carcinoma',
               'AK': 'Actinic keratosis', 'BKL': 'Benign keratosis-like lesion', 'DF': 'Dermatofibroma',
               'VASC': 'Vascular lesion', 'SCC': 'Squamous cell carcinoma'}
MALIGNANT = ['MEL', 'BCC', 'SCC']                  # the report's malignancy score = P(MEL) + P(BCC) + P(SCC)
IMG_CACHE, IMG_SIZE = 256, 224                     # stored once at 256 x 256, model input 224 x 224
EPOCHS, PATIENCE, WARMUP_EPOCHS = 15, 4, 1
BATCH = 64                                          # MobileNetV3-Large used batch 64 with no accumulation
LR_BACKBONE, LR_HEAD, WEIGHT_DECAY, GRAD_CLIP = 1e-4, 1e-3, 1e-2, 1.0
BOOTSTRAP = 2000
THREADS = max(2, min(12, os.cpu_count() or 2))
if TEST_MODE:
    EPOCHS, PATIENCE, BOOTSTRAP, THREADS = 2, 1, 100, 2

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
PROJECT = Path('/content/mnv3_demo') if IN_COLAB else Path.cwd()
DATA, RUN, EXPORT = PROJECT / 'data', PROJECT / 'run', PROJECT / 'phone_demo_export'
CACHE = DATA / 'cache256'
for p in (DATA, CACHE, RUN, EXPORT):
    p.mkdir(parents=True, exist_ok=True)


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


set_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
torch.backends.cudnn.benchmark = True

print(f'Python {platform.python_version()} | torch {torch.__version__} | torchvision {torchvision.__version__} | {platform.system()}')
print('Project folder:', PROJECT)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
elif not TEST_MODE:
    raise Stop('No CUDA GPU found. In VS Code pick the Python environment with the CUDA build of PyTorch '
               '(see README.txt), then run again.')
free_gb = shutil.disk_usage(PROJECT).free / 1e9
print(f'Free disk here: {free_gb:.0f} GB (about 25 GB needed)')
if free_gb < 25 and not TEST_MODE:
    print('WARNING: less than 25 GB free. The ISIC download (9 GB) and the resized images may not fit.')

## 2. Get the data
Both datasets download automatically, and a stopped download resumes.

**ISIC 2019** comes from the official ISIC link (about 9 GB). The images are read straight from the zip, so it is never unpacked.

**PAD-UFES-20** comes from its official Mendeley Data files (`metadata.csv` and three image zips, about 3.6 GB). Each file's size is checked. If that download fails, get it by hand: https://data.mendeley.com/datasets/zr7vgbcyr2/1 → *Download All*. Leave the zip in your Downloads folder or in the `data` folder here, then rerun.

In [ ]:
ISIC_DIR = DATA / 'isic2019'
ISIC_URL = 'https://isic-archive.s3.amazonaws.com/challenges/2019/'
ISIC_FILES = ['ISIC_2019_Training_GroundTruth.csv', 'ISIC_2019_Training_Metadata.csv', 'ISIC_2019_Training_Input.zip']
ISIC_DIR.mkdir(parents=True, exist_ok=True)


def download(url, dst):
    """Downloads url to dst. A stopped download resumes; a finished one is skipped."""
    dst.parent.mkdir(parents=True, exist_ok=True)
    part = dst.with_name(dst.name + '.part')
    have = part.stat().st_size if part.exists() else 0
    headers = {'User-Agent': 'Mozilla/5.0 (compatible; MAAI901-demo-notebook)'}
    if have:
        headers['Range'] = f'bytes={have}-'
    req = urllib.request.Request(url, headers=headers)
    try:
        resp = urllib.request.urlopen(req, timeout=60)
    except urllib.error.HTTPError as e:
        if e.code == 416 and have:              # the .part file was already complete
            os.replace(part, dst)
            return
        raise
    with resp:
        ctype = (resp.headers.get('Content-Type') or '').split(';')[0].strip().lower()
        if ctype in ('text/html', 'application/xhtml+xml'):   # a 'this file has moved' page, not the file
            head = resp.read(4096).decode('utf-8', 'replace')
            moved = ''
            if 'url=' in head:
                tail = head.split('url=', 1)[1]
                for ch in ('"', "'", '>', ' ', '\n', '\r'):
                    tail = tail.split(ch)[0]
                if tail:
                    moved = f' It now points to {tail}'
            raise Stop(f'{dst.name}: the server sent a web page, not the file, so the download address has '
                       f'changed.{moved} Update the URL in section 2 and run this cell again. Asked for: {url}')
        if have and resp.status != 206:          # the server ignored the resume request: start again
            have = 0
        total = have + int(resp.headers.get('Content-Length') or 0)
        t0, shown = time.time(), have
        with open(part, 'ab' if have else 'wb') as f:
            while True:
                chunk = resp.read(8 << 20)
                if not chunk:
                    break
                f.write(chunk)
                have += len(chunk)
                if have - shown >= 500e6:
                    shown = have
                    print(f'  {have / 1e9:.1f} of {total / 1e9:.1f} GB  ({(time.time() - t0) / 60:.1f} min)', flush=True)
    if total and part.stat().st_size != total:
        raise Stop(f'{dst.name} is incomplete. Run this cell again to resume the download.')
    os.replace(part, dst)


for name in ISIC_FILES:
    dst = ISIC_DIR / name
    if dst.exists():
        print('Already downloaded:', name)
        continue
    print('Downloading', name, '...', flush=True)
    try:
        download(ISIC_URL + name, dst)
    except Stop:
        raise
    except Exception as e:
        raise Stop(f'Could not download {name} ({type(e).__name__}: {e}). Check the internet connection and run this cell again; it resumes.')

ISIC_ZIP = ISIC_DIR / 'ISIC_2019_Training_Input.zip'
try:
    with zipfile.ZipFile(ISIC_ZIP) as z:
        isic_members = {Path(n).stem: n for n in z.namelist()
                        if n.lower().endswith('.jpg') and '__MACOSX' not in n and not Path(n).name.startswith('._')}
except zipfile.BadZipFile:
    raise Stop(f'{ISIC_ZIP.name} is damaged. Delete it and run this cell again.')
print(f'ISIC 2019: {len(isic_members):,} images in the zip (expected 25,331)')

# ---- PAD-UFES-20 ----
PAD_DIR = DATA / 'pad_ufes_20'
PAD_PAGE = 'https://data.mendeley.com/datasets/zr7vgbcyr2/1'
PAD_EXPECTED = 100 if TEST_MODE else 2298
PAD_URL = 'https://data.mendeley.com/public-files/datasets/zr7vgbcyr2/files/{}/file_downloaded'
PAD_FILES = {                     # official Mendeley Data files of PAD-UFES-20 v1: file id and exact size in bytes
    'metadata.csv': ('fa850265-57da-48f0-ba3e-998b3e44b1f6', 316209),
    'imgs_part_1.zip': ('1cc2f71f-20a2-412d-b746-a9b9bc20c966', 1245184680),
    'imgs_part_2.zip': ('559a60ed-5504-475d-996c-6a8bc253b5e7', 1126646990),
    'imgs_part_3.zip': ('34dcdf8e-e5f1-4b35-aa0b-5135051ff852', 1220565093),
}


def looks_like_pad(zpath):
    try:
        with zipfile.ZipFile(zpath) as z:
            names = [n.lower() for n in z.namelist()]
    except (zipfile.BadZipFile, OSError):
        return False
    return any(n.endswith('metadata.csv') for n in names) and any('imgs_part' in n or n.endswith('.png') for n in names)


def find_pad_zip():
    places = [DATA, PROJECT, Path.home() / 'Downloads']
    if IN_COLAB:
        places += [Path('/content'), Path('/content/drive/MyDrive')]
    for folder in places:
        if folder.is_dir():
            for zp in sorted(folder.glob('*.zip')):
                if not zp.name.startswith('ISIC_2019') and looks_like_pad(zp):
                    return zp
    return None


def pad_pngs():
    return {p.name: p for p in PAD_DIR.rglob('*.png') if '__MACOSX' not in p.parts and not p.name.startswith('._')}


def pad_metadata():
    found = [p for p in PAD_DIR.rglob('metadata.csv') if '__MACOSX' not in p.parts]
    return found[0] if found else None


if pad_metadata() is None or len(pad_pngs()) < PAD_EXPECTED:
    zpath = find_pad_zip()                                   # a "Download All" zip you got by hand, if there is one
    if zpath is not None:
        print('Unpacking', zpath, '...', flush=True)
        with zipfile.ZipFile(zpath) as z:
            z.extractall(PAD_DIR)
    else:
        print('Downloading PAD-UFES-20 from Mendeley Data (about 3.6 GB) ...', flush=True)
        PAD_DL = DATA / 'pad_download'
        try:
            for name, (file_id, size) in PAD_FILES.items():
                dst = PAD_DL / name
                if dst.exists() and dst.stat().st_size == size:
                    print('Already downloaded:', name)
                    continue
                if dst.exists():
                    dst.unlink()                             # wrong size: fetch it again
                print('Downloading', name, '...', flush=True)
                download(PAD_URL.format(file_id), dst)
                if dst.stat().st_size != size:
                    raise IOError(f'{name} is {dst.stat().st_size:,} bytes, expected {size:,}')
        except Stop:
            raise
        except Exception as e:
            raise Stop(f'The automatic PAD-UFES-20 download failed ({type(e).__name__}: {e}). Download it by hand from '
                       f'{PAD_PAGE} ("Download All"), leave the zip in your Downloads folder or put it in {DATA}, then rerun.')
        PAD_DIR.mkdir(parents=True, exist_ok=True)
        shutil.copy(PAD_DL / 'metadata.csv', PAD_DIR / 'metadata.csv')
        for name in PAD_FILES:
            if name.endswith('.zip'):
                print('Unpacking', name, '...', flush=True)
                with zipfile.ZipFile(PAD_DL / name) as z:
                    z.extractall(PAD_DIR / 'images' / Path(name).stem)
    for inner in sorted(PAD_DIR.rglob('*.zip')):          # a "Download All" zip holds imgs_part_1.zip ... inside it
        with zipfile.ZipFile(inner) as z:
            z.extractall(inner.parent / inner.stem)
        inner.unlink()
PAD_META, pad_files = pad_metadata(), pad_pngs()
if PAD_META is None or len(pad_files) < PAD_EXPECTED:
    raise Stop(f'PAD-UFES-20 looks incomplete ({len(pad_files):,} images, expected {PAD_EXPECTED:,}). '
               f'Delete {PAD_DIR}, download the zip again from {PAD_PAGE} and rerun.')
print(f'PAD-UFES-20: {len(pad_files):,} images (expected 2,298)')

## 3. One table with every image
One row per image with its source, label (mapped to the 8 ISIC classes) and the lesion or patient it belongs to. PAD-UFES-20 labels are mapped as in the report: ACK→AK, NEV→NV, SEK→BKL; MEL, BCC and SCC stay the same. PAD-UFES-20 has no DF or VASC images.

In [ ]:
gt = pd.read_csv(ISIC_DIR / 'ISIC_2019_Training_GroundTruth.csv')
md = pd.read_csv(ISIC_DIR / 'ISIC_2019_Training_Metadata.csv')
if 'UNK' in gt:
    gt = gt[gt.UNK != 1]
onehot = gt[CLASSES].to_numpy()
if not (onehot.sum(1) == 1).all():
    raise Stop('ISIC ground truth: some rows do not have exactly one label.')
isic = pd.DataFrame({'uid': gt.image.values, 'source': 'ISIC 2019', 'label': np.array(CLASSES)[onehot.argmax(1)]})
isic = isic.merge(md[['image', 'lesion_id']].rename(columns={'image': 'uid'}), on='uid', how='left')
isic['patient_id'] = np.nan
missing = sorted(set(isic.uid) - set(isic_members))
if missing:
    raise Stop(f'{len(missing)} ISIC images listed in the ground truth are not in the zip, e.g. {missing[:3]}.')
isic['file'] = isic.uid.map(isic_members)

pm = pd.read_csv(PAD_META)
PAD_MAP = {'ACK': 'AK', 'BCC': 'BCC', 'MEL': 'MEL', 'NEV': 'NV', 'SCC': 'SCC', 'SEK': 'BKL'}
pad = pd.DataFrame({'uid': 'PAD_' + pm.img_id.str.replace('.png', '', regex=False),
                    'source': 'PAD-UFES-20',
                    'label': pm.diagnostic.map(PAD_MAP),
                    'lesion_id': 'PAD_lesion_' + pm.lesion_id.astype(str),
                    'patient_id': 'PAD_' + pm.patient_id.astype(str),
                    'file': pm.img_id.map(lambda n: str(pad_files[n]) if n in pad_files else '')})
if pad.label.isna().any():
    raise Stop(f'Unknown PAD-UFES-20 diagnoses: {sorted(pm.diagnostic[pad.label.isna()].unique())}')
if (pad.file == '').any():
    raise Stop(f'{int((pad.file == "").sum())} PAD-UFES-20 images listed in metadata.csv were not found, e.g. {pm.img_id[pad.file == ""].head(3).tolist()}')

images = pd.concat([isic, pad], ignore_index=True)
images['label_idx'] = images.label.map({c: i for i, c in enumerate(CLASSES)}).astype(int)
if not TEST_MODE and (len(isic), len(pad)) != (25331, 2298):
    print(f'NOTE: expected 25,331 ISIC and 2,298 PAD images, found {len(isic):,} and {len(pad):,}.')
print(f'{len(images):,} images')
print(pd.crosstab(images.label, images.source, margins=True, margins_name='Total').reindex(CLASSES + ['Total']))

## 4. One pass over every image
Each image is read once to compute a SHA-256 fingerprint (finds identical files) and a 64-bit perceptual hash (finds near-identical images). It is then squashed to 256 × 256 pixels, keeping the whole field of view, and saved as a JPEG, as in the group's notebook. This runs on several CPU threads and is checkpointed, so a rerun continues where it stopped.

In [ ]:
FP_CSV = DATA / 'fingerprints.csv'
_local = threading.local()
_DCT = np.cos(np.pi * (2 * np.arange(32)[None, :] + 1) * np.arange(32)[:, None] / 64.0)


def read_bytes(file, source):
    if source == 'ISIC 2019':
        if getattr(_local, 'zip', None) is None:
            _local.zip = zipfile.ZipFile(ISIC_ZIP)        # one open zip per thread
        return _local.zip.read(file)
    return Path(file).read_bytes()


def phash_hex(img):
    g = np.asarray(img.convert('L').resize((32, 32), Image.LANCZOS), dtype=np.float64)
    low = (_DCT @ g @ _DCT.T)[:8, :8]
    return '%016x' % int(''.join('1' if b else '0' for b in (low > np.median(low)).flatten()), 2)


def process(args):
    uid, file, source = args
    rec = {'uid': uid, 'ok': False, 'sha256': '', 'phash': '', 'width': 0, 'height': 0, 'error': ''}
    try:
        data = read_bytes(file, source)
        rec['sha256'] = hashlib.sha256(data).hexdigest()
        with Image.open(io.BytesIO(data)) as im:
            rec['width'], rec['height'] = im.size
            if im.format == 'JPEG':
                im.draft('RGB', (IMG_CACHE * 2, IMG_CACHE * 2))
            im = ImageOps.exif_transpose(im).convert('RGB').resize((IMG_CACHE, IMG_CACHE), Image.BICUBIC)
        im.save(CACHE / f'{uid}.jpg', 'JPEG', quality=95)
        rec['phash'], rec['ok'] = phash_hex(im), True
    except Exception as e:
        rec['error'] = f'{type(e).__name__}: {e}'[:200]
    return rec


done = pd.read_csv(FP_CSV, dtype={'phash': str, 'sha256': str, 'error': str}) if FP_CSV.exists() else pd.DataFrame()
if len(done):
    done = done[(done.ok == True) & done.uid.isin(images.uid)]
    done = done[[(CACHE / f'{u}.jpg').exists() for u in done.uid]]
todo = images[~images.uid.isin(done.uid)] if len(done) else images
print(f'{len(todo):,} images to process ({len(done):,} already done), {THREADS} threads', flush=True)
rows, t0 = (done.to_dict('records') if len(done) else []), time.time()
if len(todo):
    with ThreadPoolExecutor(THREADS) as ex:
        for k, rec in enumerate(ex.map(process, zip(todo.uid, todo.file, todo.source)), 1):
            rows.append(rec)
            if k % 2000 == 0 or k == len(todo):
                pd.DataFrame(rows).to_csv(FP_CSV, index=False)
                print(f'  {k:,}/{len(todo):,}  ({(time.time() - t0) / 60:.1f} min)', flush=True)
fp = pd.DataFrame(rows).drop_duplicates('uid', keep='last')
fp.to_csv(FP_CSV, index=False)
print(f'Readable: {int(fp.ok.sum()):,}  |  unreadable: {int((~fp.ok.astype(bool)).sum())}')

## 5. Duplicates and groups
- **Unreadable images** are dropped.
- **Identical files** (same SHA-256) are kept and linked into one group. If their labels disagree, all copies are dropped, as in the report.
- **Near-identical images** (perceptual-hash distance ≤ 4 of 64 bits) with the same label are linked too. A cap of 40 images per group stops chains of merely similar-looking images. Nothing is deleted.
- **Groups** join every image of the same PAD-UFES-20 patient, ISIC lesion, identical file or near-identical pair, so a group never spans two splits.

In [ ]:
df = images.merge(fp[['uid', 'ok', 'sha256', 'phash']], on='uid', how='left')
excluded = df[df.ok != True].assign(reason='unreadable')
df = df[df.ok == True]
dup = df[df.duplicated('sha256', keep=False)]
conflict = dup.groupby('sha256').label.nunique()
conflict = set(conflict[conflict > 1].index)
excluded = pd.concat([excluded, df[df.sha256.isin(conflict)].assign(reason='identical file, different labels')])
df = df[~df.sha256.isin(conflict)].reset_index(drop=True)
print(f'Identical-file groups: {dup.sha256.nunique()} ({len(conflict)} with conflicting labels -> '
      f'{int(excluded.reason.str.startswith("identical").sum())} images dropped)')

parent, size = np.arange(len(df)), np.ones(len(df), dtype=int)
MAX_NEAR_GROUP = 40       # a near-identical link may not grow a group beyond this (stops look-alike chains)


def find(i):
    root = i
    while parent[root] != root:
        root = parent[root]
    while parent[i] != root:
        parent[i], i = root, parent[i]
    return root


def union(a, b, cap=None):
    ra, rb = find(a), find(b)
    if ra == rb:
        return True
    if cap and size[ra] + size[rb] > cap:
        return False
    lo, hi = min(ra, rb), max(ra, rb)
    parent[hi] = lo
    size[lo] += size[hi]
    return True


for key in ['patient_id', 'lesion_id', 'sha256']:
    for idx in df.groupby(key).indices.values():          # missing ids (NaN) are skipped
        for j in idx[1:]:
            union(idx[0], j)


def near_duplicate_pairs(H, max_dist=4):
    """All pairs within max_dist bits. Exact for max_dist <= 7: two such hashes share at least one of their 8 bytes."""
    pairs = set()
    for k in range(8):
        byte = ((H >> np.uint64(8 * k)) & np.uint64(255)).astype(np.int64)
        order = np.argsort(byte, kind='stable')
        for idx in np.split(order, np.flatnonzero(np.diff(byte[order])) + 1):
            for s in range(0, len(idx), 1024):
                rows_ = idx[s:s + 1024]
                ii, jj = np.nonzero(np.bitwise_count(H[rows_][:, None] ^ H[idx][None, :]) <= max_dist)
                pairs.update((int(a), int(b)) for a, b in zip(rows_[ii], idx[jj]) if a < b)
    return pairs


H = np.array([int(h, 16) for h in df.phash], dtype=np.uint64)
near = near_duplicate_pairs(H)
lab = df.label.to_numpy()
same = sorted((int(np.bitwise_count(H[a] ^ H[b])), a, b) for a, b in near if lab[a] == lab[b])   # closest pairs first
linked = sum(union(a, b, cap=MAX_NEAR_GROUP) for _, a, b in same)
df['group_id'] = [find(i) for i in range(len(df))]
gsize = df.group_id.value_counts()
print(f'Near-identical pairs: {len(near):,}. Same label: {len(same):,} ({linked:,} linked, '
      f'{len(same) - linked:,} skipped by the {MAX_NEAR_GROUP}-image cap). Different labels: {len(near) - len(same):,}, left unlinked.')
print(f'{len(df):,} images kept in {df.group_id.nunique():,} groups; largest group: {gsize.max()} images; '
      f'{len(excluded)} images dropped')

## 6. Split 70 / 15 / 15
scikit-learn's `StratifiedGroupKFold` deals whole groups into 20 folds while keeping each source × class mix balanced. Three folds become the test split, three the validation split and fourteen the training split (seed 42). The checks below confirm that no patient, lesion, identical file or group crosses splits.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

strata = (df.source + '|' + df.label).to_numpy()
fold = np.full(len(df), -1)
sgkf = StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=SEED)
for k, (_, idx) in enumerate(sgkf.split(np.zeros(len(df)), strata, groups=df.group_id.to_numpy())):
    fold[idx] = k
df['split'] = np.where(fold < 3, 'test', np.where(fold < 6, 'val', 'train'))
for key in ['group_id', 'patient_id', 'lesion_id', 'sha256']:
    crossing = df.dropna(subset=[key]).groupby(key).split.nunique()
    assert (crossing == 1).all(), f'{key} crosses splits'
print('Leakage checks passed: no group, patient, lesion or identical file is in two splits.\n')
counts = pd.crosstab(df.label, df.split).reindex(CLASSES)[['train', 'val', 'test']]
counts.loc['Total'] = counts.sum()
print(counts, '\n')
print(pd.crosstab(df.source, df.split)[['train', 'val', 'test']], '\n')
train_df, val_df, test_df = (df[df.split == s].reset_index(drop=True) for s in ('train', 'val', 'test'))
n_per_class = train_df.label_idx.value_counts().reindex(range(len(CLASSES)), fill_value=0).to_numpy()
if (n_per_class == 0).any():
    raise Stop('A class has no training images.')
CLASS_WEIGHTS = (len(train_df) / (len(CLASSES) * n_per_class)).astype(np.float32)   # from the training split only
print('Class weights:', dict(zip(CLASSES, CLASS_WEIGHTS.round(2))))
df.to_csv(RUN / 'splits.csv', index=False)
excluded.to_csv(RUN / 'excluded_images.csv', index=False)

## 7. Model and data loading
- **Model:** ImageNet-pretrained MobileNetV3-Large (torchvision, `IMAGENET1K_V2`). Its classifier is replaced by a new 8-class linear layer on the 960 pooled features, as in the group's notebook.
- **Training images:** random resized crop (60–100% of the area), horizontal and vertical flips, rotation up to 30° (half the time) and colour jitter.
- **Evaluation images:** only resized to 224 × 224.
- **Loading:** batches are built on several CPU threads. This works the same on Windows, Linux and Colab, with no extra processes.

In [ ]:
WEIGHTS = torchvision.models.MobileNet_V3_Large_Weights.IMAGENET1K_V2
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)      # ImageNet statistics used by these weights
TRAIN_TF = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0), ratio=(0.9, 1.1), interpolation=InterpolationMode.BICUBIC, antialias=True),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomApply([T.RandomRotation(30)], p=0.5),
    T.ColorJitter(0.2, 0.2, 0.2, 0.02), T.ToTensor(), T.Normalize(MEAN, STD)])
EVAL_TF = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE), interpolation=InterpolationMode.BICUBIC, antialias=True),
                     T.ToTensor(), T.Normalize(MEAN, STD)])


class Batches:
    """Yields (images, labels) batches; each batch is prepared on several threads while the GPU works on the previous one."""
    def __init__(self, frame, train, batch_size):
        self.files = [CACHE / f'{u}.jpg' for u in frame.uid]
        self.y = frame.label_idx.to_numpy()
        self.train, self.bs, self.tf, self.epoch = train, batch_size, (TRAIN_TF if train else EVAL_TF), 0

    def __len__(self):
        return len(self.y) // self.bs if self.train else math.ceil(len(self.y) / self.bs)

    def _load(self, i):
        with Image.open(self.files[i]) as im:
            return self.tf(im.convert('RGB'))

    def __iter__(self):
        order = np.arange(len(self.y))
        if self.train:                                         # new shuffle every epoch, drop the last partial batch
            np.random.default_rng(SEED * 1000 + self.epoch).shuffle(order)
            self.epoch += 1
        chunks = [order[k:k + self.bs] for k in range(0, len(self) * self.bs, self.bs)]
        with ThreadPoolExecutor(THREADS) as pool:
            nxt = [pool.submit(self._load, i) for i in chunks[0]] if chunks else []
            for c, chunk in enumerate(chunks):
                cur = nxt
                if c + 1 < len(chunks):
                    nxt = [pool.submit(self._load, i) for i in chunks[c + 1]]
                yield torch.stack([f.result() for f in cur]), torch.as_tensor(self.y[chunk])


class Net(nn.Module):
    """MobileNetV3-Large backbone + new 8-class linear head (same structure as the group's notebook)."""
    def __init__(self, pretrained=True):
        super().__init__()
        m = torchvision.models.mobilenet_v3_large(weights=WEIGHTS if pretrained else None)
        dim = m.classifier[0].in_features                     # 960
        m.classifier = nn.Identity()
        self.backbone = m
        self.head = nn.Linear(dim, len(CLASSES))
        nn.init.normal_(self.head.weight, std=0.01)
        nn.init.zeros_(self.head.bias)

    def forward(self, x):
        return self.head(self.backbone(x))


net = Net(pretrained=not TEST_MODE).to(DEVICE)
x, y = next(iter(Batches(train_df.head(8), True, 4)))
with torch.no_grad():
    net.eval()
    assert net(x.to(DEVICE)).shape == (4, len(CLASSES))
print(f'MobileNetV3-Large ready: {sum(p.numel() for p in net.parameters()) / 1e6:.2f} M parameters | '
      f'{len(train_df):,} train / {len(val_df):,} val / {len(test_df):,} test images')

## 8. Train
These are the report's settings:
- **Optimiser:** AdamW, weight decay 0.01, learning rate 10⁻⁴ for the backbone and 10⁻³ for the new head.
- **Schedule:** one epoch of linear warm-up, then cosine decay to 1% of the peak.
- **Loss:** class-weighted cross-entropy.
- **Other:** batch 64, gradient clipping at 1.0, mixed precision.
- **Stopping:** at most 15 epochs, stopping after 4 epochs without improvement in validation macro-F1. The best epoch's weights are kept.

In [ ]:
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, precision_recall_fscore_support,
                             confusion_matrix, roc_auc_score)


def compute_metrics(y_true, probs):
    """Same definitions as the group's notebook (macro scores over all 8 classes)."""
    y_true, probs = np.asarray(y_true), np.asarray(probs, dtype=np.float64)
    y_pred, labels = probs.argmax(1), list(range(len(CLASSES)))
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    auc = {CLASSES[c]: float(roc_auc_score(y_true == c, probs[:, c])) for c in labels if 0 < (y_true == c).sum() < len(y_true)}
    mel = CLASSES.index('MEL')
    tp, fn, fp_ = cm[mel, mel], cm[mel].sum() - cm[mel, mel], cm[:, mel].sum() - cm[mel, mel]
    tn = cm.sum() - tp - fn - fp_
    return {'n': int(len(y_true)), 'accuracy': float(accuracy_score(y_true, y_pred)),
            'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
            'macro_f1': float(f1_score(y_true, y_pred, labels=labels, average='macro', zero_division=0)),
            'weighted_f1': float(f1_score(y_true, y_pred, labels=labels, average='weighted', zero_division=0)),
            'roc_auc_macro_ovr': float(np.mean(list(auc.values()))) if auc else None,
            'melanoma_recall': float(tp / (tp + fn)) if tp + fn else None,
            'melanoma_specificity': float(tn / (tn + fp_)) if tn + fp_ else None,
            'per_class': {CLASSES[c]: {'precision': float(p[c]), 'recall': float(r[c]), 'f1': float(f[c]), 'support': int(s[c])} for c in labels},
            'roc_auc_per_class': auc, 'confusion_matrix': cm.tolist()}


@torch.no_grad()
def predict(model, batches):
    model.eval()
    out, ys = [], []
    for xb, yb in batches:
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(xb.to(DEVICE, non_blocking=True))
        out.append(F.softmax(logits.float(), 1).cpu())
        ys.append(yb)
    return torch.cat(out).numpy(), torch.cat(ys).numpy()


def weighted_nll(probs, y):
    w = CLASS_WEIGHTS[y]
    return float((w * -np.log(probs[np.arange(len(y)), y] + 1e-12)).sum() / w.sum())


train_b, val_b = Batches(train_df, True, BATCH), Batches(val_df, False, 128)
steps = len(train_b)
total_steps, warm = EPOCHS * steps, max(1, WARMUP_EPOCHS * steps)
opt = torch.optim.AdamW([{'params': net.head.parameters(), 'lr': LR_HEAD},
                         {'params': net.backbone.parameters(), 'lr': LR_BACKBONE}], weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else
                                          0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total_steps - warm))))
scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
criterion = nn.CrossEntropyLoss(weight=torch.tensor(CLASS_WEIGHTS, device=DEVICE))
BEST, LAST, HIST, DONE = RUN / 'best.pt', RUN / 'last.pt', RUN / 'history.csv', RUN / 'training_done.json'

hist, best, best_epoch, bad, start = [], -1.0, 0, 0, 1
if DONE.exists():
    info = json.loads(DONE.read_text())
    hist, best, best_epoch, start = pd.read_csv(HIST).to_dict('records'), info['best_val_macro_f1'], info['best_epoch'], EPOCHS + 1
    print(f'Training already finished (best epoch {best_epoch}). Delete the "run" folder to train again.')
elif LAST.exists():
    ck = torch.load(LAST, map_location='cpu', weights_only=False)
    net.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); scaler.load_state_dict(ck['scaler'])
    sched.load_state_dict(ck['sched'])
    hist, best, best_epoch, bad, start = ck['hist'], ck['best'], ck['best_epoch'], ck['bad'], ck['epoch'] + 1
    train_b.epoch = ck['epoch']
    print(f'Resuming at epoch {start} (best so far {best:.4f} at epoch {best_epoch})')

t_train = time.time()
for epoch in range(start, EPOCHS + 1):
    if bad >= PATIENCE:
        break
    t0 = time.time()
    net.train()
    run_loss, nb = torch.zeros((), device=DEVICE), 0
    for xb, yb in train_b:
        xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = net(xb)
        loss = criterion(logits.float(), yb)
        if not torch.isfinite(loss):
            raise Stop(f'The loss became NaN/inf at epoch {epoch}.')
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(net.parameters(), GRAD_CLIP)
        scaler.step(opt)
        scaler.update()
        sched.step()
        run_loss += loss.detach()
        nb += 1
    probs, yv = predict(net, val_b)
    mv = compute_metrics(yv, probs)
    hist.append({'epoch': epoch, 'train_loss': float(run_loss / max(nb, 1)), 'val_loss': weighted_nll(probs, yv),
                 'val_macro_f1': mv['macro_f1'], 'val_balanced_acc': mv['balanced_accuracy'], 'val_accuracy': mv['accuracy'],
                 'lr_backbone': opt.param_groups[1]['lr'], 'seconds': round(time.time() - t0, 1)})
    if mv['macro_f1'] > best + 1e-4:
        best, best_epoch, bad = mv['macro_f1'], epoch, 0
        torch.save({'model': net.state_dict(), 'epoch': epoch, 'val_macro_f1': best}, BEST)
    else:
        bad += 1
    pd.DataFrame(hist).to_csv(HIST, index=False)
    torch.save({'model': net.state_dict(), 'opt': opt.state_dict(), 'scaler': scaler.state_dict(), 'sched': sched.state_dict(),
                'hist': hist, 'best': best, 'best_epoch': best_epoch, 'bad': bad, 'epoch': epoch}, LAST)
    h = hist[-1]
    print(f'epoch {epoch:2d}/{EPOCHS}  train loss {h["train_loss"]:.3f}  val loss {h["val_loss"]:.3f}  '
          f'val macro-F1 {h["val_macro_f1"]:.4f}  bal. acc {h["val_balanced_acc"]:.3f}  '
          f'best {best:.4f} @ {best_epoch}  ({h["seconds"]:.0f} s)', flush=True)

if not DONE.exists():
    DONE.write_text(json.dumps({'best_epoch': best_epoch, 'best_val_macro_f1': best, 'epochs_run': len(hist),
                                'minutes_this_session': round((time.time() - t_train) / 60, 1),
                                'stopped_early': len(hist) < EPOCHS}, indent=2))
    if LAST.exists():
        LAST.unlink()
print(f'\nBest validation macro-F1 {best:.4f} at epoch {best_epoch} | total training time '
      f'{sum(h["seconds"] for h in hist) / 60:.1f} min')

## 9. Evaluate on the test split
The test split is used once, with the best checkpoint. Thresholds are chosen on the validation split, as in the report:
- **Melanoma:** the probability threshold that gives 90% specificity on validation.
- **"Refer" flag:** the malignancy score, P(MEL) + P(BCC) + P(SCC), at 90% specificity on validation.

95% confidence intervals come from 2,000 group-bootstrap resamples of the test split, so a patient's or lesion's images stay together.

In [ ]:
from scipy.stats import rankdata

net.load_state_dict(torch.load(BEST, map_location='cpu', weights_only=False)['model'])
val_probs, yv = predict(net, Batches(val_df, False, 128))
test_probs, yt = predict(net, Batches(test_df, False, 128))
m = compute_metrics(yt, test_probs)

K, MEL = len(CLASSES), CLASSES.index('MEL')
MAL = [CLASSES.index(c) for c in MALIGNANT]
mel_thr = {t: float(np.quantile(val_probs[yv != MEL, MEL], t)) for t in (0.90, 0.95)}
mal_val = val_probs[:, MAL].sum(1)
refer_thr = float(np.quantile(mal_val[~np.isin(yv, MAL)], 0.90))
p_mel, is_mel = test_probs[:, MEL], yt == MEL
mal_test, is_mal = test_probs[:, MAL].sum(1), np.isin(yt, MAL)


def auc_rank(y, s):
    n1 = int(y.sum()); n0 = len(y) - n1
    return (rankdata(s)[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0) if n1 and n0 else np.nan


def f1_bal(y, pred):
    cm = np.bincount(y * K + pred, minlength=K * K).reshape(K, K)
    tp, sup, pp = np.diag(cm), cm.sum(1), cm.sum(0)
    pr = np.divide(tp, pp, out=np.zeros(K), where=pp > 0)
    rc = np.divide(tp, sup, out=np.zeros(K), where=sup > 0)
    f1 = np.divide(2 * pr * rc, pr + rc, out=np.zeros(K), where=(pr + rc) > 0)
    return f1.mean(), rc[sup > 0].mean()


def stats(ix):
    mf1, bal = f1_bal(yt[ix], test_probs[ix].argmax(1))
    return {'macro_f1': mf1, 'balanced_accuracy': bal, 'mel_auc': auc_rank(is_mel[ix], p_mel[ix]),
            'mel_sens_at_90spec': float((p_mel[ix][is_mel[ix]] > mel_thr[0.90]).mean()) if is_mel[ix].any() else np.nan,
            'malignancy_auc': auc_rank(is_mal[ix], mal_test[ix]),
            'refer_sensitivity': float((mal_test[ix][is_mal[ix]] > refer_thr).mean()) if is_mal[ix].any() else np.nan,
            'refer_specificity': float((mal_test[ix][~is_mal[ix]] <= refer_thr).mean()) if (~is_mal[ix]).any() else np.nan}


point = stats(np.arange(len(yt)))
codes = pd.factorize(test_df.group_id)[0]
members = np.split(np.argsort(codes, kind='stable'), np.cumsum(np.bincount(codes))[:-1])
rng = np.random.default_rng(SEED)
boot = pd.DataFrame([stats(np.concatenate([members[k] for k in rng.integers(0, len(members), len(members))]))
                     for _ in range(BOOTSTRAP)])
ci = {k: (float(np.nanpercentile(boot[k], 2.5)), float(np.nanpercentile(boot[k], 97.5))) for k in point}

# The group's MobileNetV3-Large (full fine-tune) on its own test split, as reported
REPORT = {'Accuracy': (0.752, None), 'Balanced accuracy': (0.652, (0.619, 0.685)), 'Macro-F1': (0.642, (0.612, 0.669)),
          'Weighted-F1': (0.748, None), 'Macro ROC-AUC': (0.940, None), 'Melanoma AUC': (0.892, (0.878, 0.907)),
          'Melanoma sensitivity at 90% specificity': (0.659, (0.620, 0.700)),
          'Malignancy AUC (MEL+BCC+SCC)': (0.905, (0.894, 0.915))}
OURS = {'Accuracy': (m['accuracy'], None), 'Balanced accuracy': (point['balanced_accuracy'], ci['balanced_accuracy']),
        'Macro-F1': (point['macro_f1'], ci['macro_f1']), 'Weighted-F1': (m['weighted_f1'], None),
        'Macro ROC-AUC': (m['roc_auc_macro_ovr'], None), 'Melanoma AUC': (point['mel_auc'], ci['mel_auc']),
        'Melanoma sensitivity at 90% specificity': (point['mel_sens_at_90spec'], ci['mel_sens_at_90spec']),
        'Malignancy AUC (MEL+BCC+SCC)': (point['malignancy_auc'], ci['malignancy_auc'])}
fmt = lambda v: '' if v[0] is None else f'{v[0]:.3f}' + (f' [{v[1][0]:.3f}, {v[1][1]:.3f}]' if v[1] else '')
cmp_table = pd.DataFrame({'This model (demo)': {k: fmt(v) for k, v in OURS.items()},
                          'Report (group run)': {k: fmt(v) for k, v in REPORT.items()}})
print(f'Test split: {len(yt):,} images, {test_df.group_id.nunique():,} groups; best epoch {best_epoch}\n')
print(cmp_table.to_string(), '\n')
print('The two models were tested on different (but equally built) test splits, so small differences are expected.\n')
per_class = pd.DataFrame(m['per_class']).T[['precision', 'recall', 'f1', 'support']]
per_class['support'] = per_class['support'].astype(int)
print('Per class (test):'); print(per_class.round(3).to_string(), '\n')
print(f'"Refer" flag on the test split (threshold {refer_thr:.3f}, chosen on validation): '
      f'sensitivity {point["refer_sensitivity"]:.3f} [{ci["refer_sensitivity"][0]:.3f}, {ci["refer_sensitivity"][1]:.3f}], '
      f'specificity {point["refer_specificity"]:.3f} [{ci["refer_specificity"][0]:.3f}, {ci["refer_specificity"][1]:.3f}]')

metrics_out = {'test': m, 'test_point': point, 'test_ci95': ci, 'thresholds': {'melanoma_90spec': mel_thr[0.90],
               'melanoma_95spec': mel_thr[0.95], 'refer_malignancy_90spec': refer_thr},
               'best_epoch': best_epoch, 'best_val_macro_f1': best, 'history': hist,
               'n_images': {'train': len(train_df), 'val': len(val_df), 'test': len(test_df), 'excluded': len(excluded)}}
(EXPORT / 'metrics.json').write_text(json.dumps(metrics_out, indent=2, default=float))

In [ ]:
FIG = EXPORT / 'figures'
FIG.mkdir(exist_ok=True)
H_ = pd.DataFrame(hist)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(H_.epoch, H_.train_loss, 'o-', label='train loss')
ax[0].plot(H_.epoch, H_.val_loss, 'o-', label='validation loss')
ax[0].set_xlabel('epoch'); ax[0].legend(frameon=False); ax[0].set_title('Loss')
ax[1].plot(H_.epoch, H_.val_macro_f1, 'o-', label='validation macro-F1')
ax[1].axvline(best_epoch, color='grey', ls=':', lw=1)
ax[1].set_xlabel('epoch'); ax[1].set_title(f'Validation macro-F1 (best: epoch {best_epoch})')
fig.tight_layout(); fig.savefig(FIG / 'learning_curves.png', dpi=200); plt.show(); plt.close(fig)

cm = np.array(m['confusion_matrix'], dtype=float)
cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
fig, ax = plt.subplots(figsize=(5.6, 4.8))
im_ = ax.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(K), CLASSES); ax.set_yticks(range(K), CLASSES)
ax.set_xlabel('predicted'); ax.set_ylabel('true'); ax.set_title('Test confusion matrix (row-normalised)')
for i in range(K):
    for j in range(K):
        ax.text(j, i, f'{cmn[i, j]:.2f}', ha='center', va='center', fontsize=7, color='white' if cmn[i, j] > 0.5 else 'black')
fig.colorbar(im_, fraction=0.046); fig.tight_layout(); fig.savefig(FIG / 'confusion_matrix.png', dpi=200); plt.show(); plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(CLASSES, per_class.f1.values, color='#2a78d6')
for i, v in enumerate(per_class.f1.values):
    ax.text(i, v + 0.01, f'{v:.2f}', ha='center', fontsize=8)
ax.set_ylim(0, 1); ax.set_ylabel('F1'); ax.set_title('Test F1 per class')
fig.tight_layout(); fig.savefig(FIG / 'per_class_f1.png', dpi=200); plt.show(); plt.close(fig)

## 10. Export for the phone
The model is saved in ONNX format. A web page can run ONNX models on a phone with ONNX Runtime Web. The exported file includes the normalisation and the softmax, so the page only has to resize the photo to 224 × 224 and pass in RGB values from 0 to 1.

The export is checked against PyTorch on test images. It also includes the thresholds, a short metrics summary and a few test images for the "sample" buttons. The samples are typical correct predictions per class plus one missed melanoma, so the demo stays honest.

In [ ]:
import onnx
import onnxruntime as ort


class PhoneModel(nn.Module):
    """Input: (1, 3, 224, 224) RGB in [0, 1]. Output: the 8 class probabilities."""
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.register_buffer('mean', torch.tensor(MEAN).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor(STD).view(1, 3, 1, 1))

    def forward(self, x):
        return torch.softmax(self.model((x - self.mean) / self.std), dim=1)


phone = PhoneModel(Net(pretrained=False)).eval()
phone.model.load_state_dict(torch.load(BEST, map_location='cpu', weights_only=False)['model'])
ONNX_PATH = EXPORT / 'model.onnx'
dummy = torch.rand(1, 3, IMG_SIZE, IMG_SIZE)
try:
    torch.onnx.export(phone, (dummy,), str(ONNX_PATH), input_names=['image'], output_names=['probs'],
                      opset_version=18, dynamo=True, external_data=False)
except Exception as e:
    print(f'New ONNX exporter failed ({type(e).__name__}: {str(e)[:150]}); using the classic exporter.')
    torch.onnx.export(phone, (dummy,), str(ONNX_PATH), input_names=['image'], output_names=['probs'],
                      opset_version=17, dynamo=False)
onnx_model = onnx.load(str(ONNX_PATH))                      # also pulls in weights if they were saved separately
if onnx_model.ir_version > 10:
    onnx_model.ir_version = 10      # the phone runtime (ONNX Runtime Web 1.22) reads up to IR 10; this CNN uses nothing newer
onnx.save_model(onnx_model, str(ONNX_PATH), save_as_external_data=False)
for extra in EXPORT.glob('model.onnx.data*'):
    extra.unlink()
onnx.checker.check_model(str(ONNX_PATH))

sess = ort.InferenceSession(str(ONNX_PATH), providers=['CPUExecutionProvider'])
in_name, out_name = sess.get_inputs()[0].name, sess.get_outputs()[0].name
to01 = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE), interpolation=InterpolationMode.BICUBIC, antialias=True), T.ToTensor()])
check = test_df.sample(min(64, len(test_df)), random_state=SEED)
diffs, agree = [], 0
for u in check.uid:
    with Image.open(CACHE / f'{u}.jpg') as im:
        x01 = to01(im.convert('RGB')).unsqueeze(0)
    with torch.no_grad():
        p_t = phone(x01).numpy()
    p_o = sess.run([out_name], {in_name: x01.numpy()})[0]
    diffs.append(float(np.abs(p_t - p_o).max()))
    agree += int(p_t.argmax() == p_o.argmax())
print(f'ONNX vs PyTorch on {len(check)} test images: largest probability difference {max(diffs):.2e}, '
      f'same top class {agree}/{len(check)}')
assert max(diffs) < 1e-3 and agree == len(check), 'The exported model does not match PyTorch'

# Sample images for the phone page: from the TEST split only (never seen in training)
pred, conf = test_probs.argmax(1), test_probs.max(1)
picked = []
for c in ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'SCC']:
    ci_ = CLASSES.index(c)
    for src in (['PAD-UFES-20', 'ISIC 2019'] if c in ('AK', 'BCC') else ['ISIC 2019']):   # include phone photos (PAD)
        ok = np.flatnonzero((yt == ci_) & (pred == ci_) & (test_df.source.to_numpy() == src))
        if len(ok):
            picked.append((ok[np.argsort(conf[ok])[len(ok) // 2]], 'typical correct prediction'))   # median confidence
            break
missed = np.flatnonzero((yt == MEL) & (pred != MEL))
if len(missed):
    picked.append((missed[np.argsort(conf[missed])[len(missed) // 2]], 'melanoma the model missed'))
SAMPLES = EXPORT / 'samples'
if SAMPLES.exists():
    shutil.rmtree(SAMPLES)
SAMPLES.mkdir()
samples = []
for n, (i, why) in enumerate(picked, 1):
    r = test_df.iloc[i]
    with Image.open(io.BytesIO(read_bytes(images.set_index('uid').file[r.uid], r.source))) as im:
        im = ImageOps.exif_transpose(im).convert('RGB')
        im.thumbnail((512, 512), Image.LANCZOS)
        fname = f'{n:02d}_{r.label}_{"PAD" if r.source == "PAD-UFES-20" else "ISIC"}.jpg'
        im.save(SAMPLES / fname, 'JPEG', quality=88)
    samples.append({'file': f'samples/{fname}', 'true_label': r.label, 'source': r.source, 'note': why,
                    'model_top': CLASSES[int(pred[i])], 'model_top_prob': round(float(conf[i]), 3), 'image_id': r.uid})
print(f'{len(samples)} sample images:', ', '.join(s['file'].split('/')[1] for s in samples))

config = {
    'model': 'MobileNetV3-Large, full fine-tune (MAAI901 group recipe), demo retrain',
    'onnx_file': 'model.onnx', 'input_name': in_name, 'output_name': out_name,
    'input': {'size': IMG_SIZE, 'layout': 'NCHW float32, RGB in [0, 1]',
              'resize': 'squash the whole photo to a square (no crop), as in training'},
    'classes': CLASSES, 'class_names': CLASS_NAMES, 'malignant_classes': MALIGNANT,
    'refer_threshold': refer_thr, 'melanoma_threshold_90spec': mel_thr[0.90],
    'test_summary': {'n_test_images': int(len(yt)), 'macro_f1': point['macro_f1'], 'macro_f1_ci95': ci['macro_f1'],
                     'balanced_accuracy': point['balanced_accuracy'], 'melanoma_auc': point['mel_auc'],
                     'malignancy_auc': point['malignancy_auc'], 'refer_sensitivity': point['refer_sensitivity'],
                     'refer_specificity': point['refer_specificity']},
    'samples': samples,
    'data_credits': 'ISIC 2019 (CC BY-NC 4.0; Codella et al., 2018; Tschandl et al., 2018; Hernandez-Perez et al., 2024) '
                    'and PAD-UFES-20 (CC BY 4.0; Pacheco et al., 2020)',
    'created': time.strftime('%Y-%m-%d %H:%M'),
    'versions': {'torch': torch.__version__, 'torchvision': torchvision.__version__, 'onnx': onnx.__version__,
                 'onnxruntime': ort.__version__, 'python': platform.python_version()}}
(EXPORT / 'demo_config.json').write_text(json.dumps(config, indent=2, default=float))
zip_path = shutil.make_archive(str(PROJECT / 'phone_demo_export'), 'zip', EXPORT)
print(f'\nmodel.onnx: {ONNX_PATH.stat().st_size / 1e6:.1f} MB')
print(f'DONE. {zip_path} ({Path(zip_path).stat().st_size / 1e6:.1f} MB) is everything the phone page needs.')